In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import brainshare as br

In [9]:
br.sign_in()

hello brainworld


In [27]:
br.set_project('default')

In [10]:
br.query("*")
# 

# Design approach

- minimize magic so this we can port to multiple languages (esp. ts)
- minimize string parsing
- equivalents of nestable SQL & Cypher expressions are nested python function calls

In [5]:
# TODO in neo4j, nodes have multiple labels. should 
# we allow nodes with multiple node_types?
# neo4j uses these for indexing, but we can easily 
# layer in additional indexes. We use types for 
# defining uniqueness across tenants
#
# neo4j: A relationship must have a start node, an end node,
# and exactly one type. 

In [9]:
# TODO if the brainshare query needs to run as a graph query,
# where do we generate that query, and how do we run it safely?
# There's a reason web apps frontends don't generate SQL.
# Maybe we can get by with a cypher-esque intermediate
# represenation. 

In [11]:
# TODO should translate br queries to postgrest or graphql?

In [6]:
# our relational query does not look like SQL because
# we don't have a SQL interpreter. same can apply to
# graph queries: no need for a full cypher interpreter
# for this api to be useful. on the contrary, a simplified
# query language is acceptible as long as we don't overextend
# a design for the usecase (here, graph -> table for simple
# scenarios). anything fancier by exporting data to your
# favorite DB.

In [7]:
# query examples

### SQL:

```sql
SELECT * FROM chemical JOIN chemical_synonym ...;
```

### PostgREST:

```typescript
.from("chemical").select("*, synonym(*), reaction(*), chemical_history(*, profile(*))")
```

### Cypher:

```cypher
MATCH
````

### SQLAlchemy

```python
```

### pandas

```python
df = chemical.merge(synonym, on="chemical_id")
```

### polars

```python
df = 
```

### R

```R
```

### Brainshare

- returns multiples dataframes or 1 graph or nested json
- If we introspect on the database, we can generate types,
  but mypy won't know about them
- TODO how closely can we match the Polars API?

```python
select("*").join(right="synonym") # implicit from="chemical"
    .join("synonym", "reaction", left_on="reaction_id", right_on="id")
    .join("reaction", "reaction_history")
    .join("chemical", "synonym", right_alias="synonym2")
```


### Aggregate function with *

#### SQL:

```sql
SELECT count(*), order_date FROM orders GROUP BY order_date;
```

#### PostgREST:

```http
GET /orders?select=count(),order_date HTTP/1.1
```

#### Cypher:

```cypher

````

#### SQLAlchemy

```python
```

#### pandas, polars, R

N/A

#### Brainshare

?

TODO consider how sqlalchemy uses `select_from` to achieve:

> A typical example:
> ```python
>   q = session.query(Address).select_from(User).\
>       join(User.addresses).\
>       filter(User.name == 'ed')
> ```
>         
> Which produces SQL equivalent to:
> 
> ```SQL
> SELECT address.* FROM user
> JOIN address ON user.id=address.user_id
> WHERE user.name = :name_1
> ```

### Brainshare

We can do this to keep the API smaller:

```python
br.select("address.*").join("user", "address")
  .where(br.eq("user.name", name))
# or shortcut version:
br.select("address.*").join("user", "address")
  .eq("user.name", name)
```

# SQL:
# SELECT * FROM chemical WHERE chemical.name = "g3p";
#
# PostgREST:
# .from("chemical").match({ name: "g3p" })
#
# Cypher:
# MATCH :Chemical ....
#
# Brainshare: (returns multiples dataframes or 1 graph or nested json)
# .query("chemical(*). ")

In [9]:
model = br.query('*.model="e-coli-core').to_graph()

AttributeError: 'Query' object has no attribute 'to_graph'

In [6]:
rxn_df = br.query('type="reaction"').to_dataframe()

In [8]:
rxn_df

,id,name
0,1,test


In [12]:
# TODO match the pandas API for merging
# what about filtering? pandas doesn't have a method-based filtering
# API, which we'd really need. Can't do SQL because we don't have an
# interpreter; same for Cypher (we'd export to bigquery, etc. for that).
# Something like the supabase API is nice, but it needs to be instantly
# familiar to comp scientists who would know pandas/polars/R + maybe sql.
# 
# Polars is a good target because it's newer so less sprawling, with the
# common features all supported.
#
# refs:
# https://docs.pola.rs/user-guide/basics/expressions/#select-statement
# https://docs.pola.rs/py-polars/html/reference/dataframe/api/polars.DataFrame.filter.html
join_df = br.query(type="reaction", join="chemical").to_dataframe()

In [13]:
join_df

,id,name
0,1,test


In [ ]:
# for graphs, there isn't an equivalent of cypher that's used regularly
# by scientists. For a query like "give me all articles that reference
# a reaction that consumes Pyruvate with Delta G > 0",  you could:
# - pull in all the reactions as dataframe, joined to article & chem
# - implement a cypher query that does the work, using LLM assistance, but
#
# related and neat:
# https://github.com/emehrkay/Pypher
